# Module 10: Outfit Recommendation Engine
## Multi-Item Outfit Generation, Beam Search & Cohesion Scoring

This notebook demonstrates:
1. Loading the `OutfitRecommender` engine.
2. Generating complete looks from a seed item (Top, Bottom, Shoes, Accessory).
3. Generating full outfits based on occasion and demographic prompts.
4. Calculating global pairwise outfit cohesion scores.
5. Interactive item substitution preserving maximal stylistic harmony.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns

from src.embeddings import EmbeddingManager
from src.style_matcher import StyleMatcher
from src.recommender import OutfitRecommender

sns.set_theme(style="white", palette="muted")
plt.rcParams["figure.figsize"] = (15, 6)

### 1. Initialize Outfit Recommender Engine

In [ ]:
mgr = EmbeddingManager()
matcher = StyleMatcher(mgr)
recommender = OutfitRecommender(style_matcher=matcher)

print(f"Catalog Size : {len(mgr):,} indexed fashion items")
print(f"Outfit Slots : {', '.join(['top', 'bottom', 'shoes', 'accessory'])}")

### 2. Complete Outfit Generation from Seed Garment

Select any garment from the catalog and assemble a complete, harmonious 4-piece look.

In [ ]:
# Select a seed garment
sample_seed = mgr.index_df[mgr.index_df["outfit_part"] == "top"].iloc[5]
seed_id = sample_seed["id"]

outfits = recommender.generate_outfit_from_seed(
    seed_item_id=seed_id,
    include_accessory=True,
    top_k_outfits=2,
)

print(f"Seed Garment: ID {seed_id} - {sample_seed.get('productDisplayName')} ({sample_seed.get('baseColour')})")
best_outfit = outfits[0]
print(f"Global Cohesion Score: {best_outfit['cohesion_score']:.3f}")
print(f"Color Palette: {best_outfit['color_palette']}")
for item in best_outfit["items"]:
    print(f"  [{item['outfit_part'].upper():9s}] {item['canonical_category']} | {item['baseColour']} | {item['productDisplayName']}")

### 3. Visualizing the Complete Recommended Look

In [ ]:
fig, axes = plt.subplots(1, len(best_outfit["items"]), figsize=(18, 5))

for i, item in enumerate(best_outfit["items"]):
    img = Image.open(item["image_path"]).convert("RGB")
    axes[i].imshow(img)
    role_badge = item["outfit_part"].upper()
    is_seed = (str(item["id"]) == str(seed_id))
    title_color = "darkred" if is_seed else "navy"
    seed_tag = "(SEED ITEM)\n" if is_seed else ""
    
    axes[i].set_title(
        f"{seed_tag}[{role_badge}]\n{item['productDisplayName'][:28]}\n{item['canonical_category']} - {item['baseColour']}",
        fontsize=10,
        color=title_color,
        fontweight="bold",
    )
    axes[i].axis("off")

plt.suptitle(f"Complete Outfit Assembly (Cohesion Score: {best_outfit['cohesion_score']:.3f})", fontsize=15, y=1.05)
plt.tight_layout()
plt.show()

### 4. Interactive Item Substitution

If the user wants an alternative option for any piece (e.g. swap footwear), the engine provides top alternatives ranked by updated outfit harmony.

In [ ]:
# Swap the footwear item
shoes_item = [it for it in best_outfit["items"] if it["outfit_part"] == "shoes"][0]

alternatives = recommender.substitute_item_in_outfit(
    outfit_items=best_outfit["items"],
    replace_item_id=shoes_item["id"],
    top_k_alternatives=3,
)

print(f"Alternative Footwear Options for ID {shoes_item['id']} ({shoes_item['productDisplayName']}):")
for rank, alt in enumerate(alternatives, 1):
    print(f"  #{rank} [New Cohesion: {alt['new_outfit_cohesion']:.3f}] - {alt['canonical_category']} ({alt['baseColour']}) | {alt['productDisplayName']}")

### 5. Occasion & Demographic Curated Outfit

In [ ]:
curated_outfits = recommender.generate_outfit_by_occasion(
    occasion="Casual",
    gender="Men",
    include_accessory=True,
    top_k_outfits=1,
)

if curated_outfits:
    outfit = curated_outfits[0]
    print(f"Curated Men's Casual Look (Cohesion: {outfit['cohesion_score']:.3f}):")
    for item in outfit["items"]:
        print(f"  * [{item['outfit_part'].upper():9s}] {item['productDisplayName']} ({item['baseColour']})")